In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [2]:
pip install transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 38.7 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ

In [3]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForMultipleChoice,TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [4]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [5]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


# EDA

In [6]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

(2000, 8)
(500, 7)


In [7]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [8]:
train_df['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [9]:
train_df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [10]:
train_df, val_df = train_test_split(
    train_df, 
    test_size=0.2, 
    stratify=train_df["answer"], 
    random_state=4524
)

# Evaluation Metric

In [11]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    
    predictions = np.argsort(logits, axis=-1)[:, ::-1][:, :3]
    
    score = 0.0
    for actual, pred in zip(labels, predictions):
        if actual == pred[0]:
            score += 1.0
        elif actual == pred[1]:
            score += 0.5
        elif actual == pred[2]:
            score += 1/3
            
    return {"map3": score / len(labels)}

# Model 1: LSTM + Transformer

In [12]:
class HybridQAModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, lstm_hidden=128, num_heads=4, num_layers=2):
        super(HybridQAModel, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, lstm_hidden, batch_first=True, bidirectional=True)
        self.layer_norm = nn.LayerNorm(lstm_hidden * 2)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=lstm_hidden * 2, 
            nhead=num_heads, 
            batch_first=True)
        
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(lstm_hidden * 4, 1)

    def forward(self, input_ids):
        batch_size, num_options, seq_len = input_ids.shape
        x = input_ids.view(batch_size * num_options, seq_len)
        padding_mask = (x == 0) 
        
        x = self.embedding(x) 
        lstm_out, _ = self.lstm(x) 
        lstm_out = self.layer_norm(lstm_out)
        trans_out = self.transformer(lstm_out, src_key_padding_mask=padding_mask)
        
        avg_pool = trans_out.mean(dim=1)
        max_pool = trans_out.max(dim=1)[0]
        pooled = torch.cat((avg_pool, max_pool), dim=1)
        
        logits = self.classifier(pooled) 
        logits = logits.view(batch_size, num_options)
        
        return logits

In [13]:
class MCQDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=128):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        prompt = str(row['prompt'])
        options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
        
        input_ids = []
        for opt in options:
            encoded = self.tokenizer(
                prompt, opt, 
                truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt'
            )
            input_ids.append(encoded['input_ids'].squeeze(0))
            
        label = self.label_map[row['answer']]
        
        return torch.stack(input_ids), torch.tensor(label, dtype=torch.long)

In [14]:
def train_scratch_model(train_loader, val_loader, vocab_size, epochs=5):
    wandb.init(project="24f3004524-t22026", name="Scratch-LSTM-Transformer-2")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = HybridQAModel(vocab_size=vocab_size).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_inputs, batch_labels in train_loader:
            batch_inputs, batch_labels = batch_inputs.to(device), batch_labels.to(device)
            
            optimizer.zero_grad()
            logits = model(batch_inputs) 
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        model.eval()
        val_loss, correct, map3_score = 0, 0, 0
        
        with torch.no_grad():
            for batch_inputs, batch_labels in val_loader:
                batch_inputs, batch_labels = batch_inputs.to(device), batch_labels.to(device)
                logits = model(batch_inputs)
                
                loss = criterion(logits, batch_labels)
                val_loss += loss.item()
                
                preds = torch.argsort(logits, dim=1, descending=True)
                correct += (preds[:, 0] == batch_labels).sum().item()
                
                for i in range(len(batch_labels)):
                    actual = batch_labels[i].item()
                    top3 = preds[i, :3].tolist()
                    if actual in top3:
                        rank = top3.index(actual) + 1
                        map3_score += 1.0 / rank

        avg_train_loss = total_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / len(val_loader.dataset)
        val_map3 = map3_score / len(val_loader.dataset)
        
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_accuracy": val_acc,
            "val_map@3": val_map3
        })
        
        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | MAP@3: {val_map3:.4f}")

    wandb.finish()
    return model

In [15]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
vocab_size = tokenizer.vocab_size

train_records = train_df.to_dict('records')
val_records = val_df.to_dict('records')
test_records = test_df.to_dict('records')

train_dataset = MCQDataset(train_records, tokenizer)
val_dataset = MCQDataset(val_records, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

trained_model = train_scratch_model(train_loader, val_loader, vocab_size, epochs=3)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

wandb: setting up run ndzcbqdv
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260709_212117-ndzcbqdv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Scratch-LSTM-Transformer-2
wandb: ⭐️ View project at https://wandb.ai/pranay12aggarwal-indian-institute-of-technology-madras/24f3004524-t22026
wandb: 🚀 View run at https://wandb.ai/pranay12aggarwal-indian-institute-of-technology-madras/24f3004524-t22026/runs/ndzcbqdv
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1 | Train Loss: 0.6173 | Val Loss: 0.1943 | MAP@3: 0.9758
Epoch 2 | Train Loss: 0.0501 | Val Loss: 0.0488 | MAP@3: 0.9950


wandb: uploading history steps 1-1, summary, console lines 3-3; updating run metadata


Epoch 3 | Train Loss: 0.0039 | Val Loss: 0.0723 | MAP@3: 0.9912


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading output.log; uploading wandb-summary.json
wandb: uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:        epoch ▁▅█
wandb:   train_loss █▂▁
wandb: val_accuracy ▁██
wandb:     val_loss █▁▂
wandb:    val_map@3 ▁█▇
wandb: 
wandb: Run summary:
wandb:        epoch 3
wandb:   train_loss 0.00391
wandb: val_accuracy 0.99
wandb:     val_loss 0.07232
wandb:    val_map@3 0.99125
wandb: 
wandb: 🚀 View run Scratch-LSTM-Transformer-2 at: https://wandb.ai/pranay12aggarwal-indian-institute-of-technology-madras/24f3004524-t22026/runs/ndzcbqdv
wandb: ⭐️ View project at: https://wandb.ai/pranay12aggarwal-indian-institute-of-technology-madras/24f3004524-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260709_212117-ndzcbqdv/logs


In [16]:
class TestMCQDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=128):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        prompt = str(row['prompt'])
        options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]

        input_ids = []
        for opt in options:
            encoded = self.tokenizer(
                prompt, opt,
                truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt'
            )
            input_ids.append(encoded['input_ids'].squeeze(0))

        return torch.stack(input_ids), row['id']

test_dataset = TestMCQDataset(test_records, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

trained_model.eval()

HybridQAModel(
  (embedding): Embedding(30522, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Linear(in_features=512, out_features=1, bias=True)
)

In [17]:
predictions = []
test_ids = []
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = trained_model.to(device)

with torch.no_grad():
    for batch_inputs, batch_ids in test_loader:
        batch_inputs = batch_inputs.to(device)
        
        logits = trained_model(batch_inputs)
        
        top3_preds = torch.argsort(logits, dim=1, descending=True)[:, :3].cpu().numpy()
        
        for i in range(len(batch_ids)):
            letters = [idx_to_letter[idx] for idx in top3_preds[i]]
            predictions.append(" ".join(letters))
            test_ids.append(batch_ids[i].item() if isinstance(batch_ids[i], torch.Tensor) else batch_ids[i])

# Submission Cell

In [18]:
submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,A E C
1,2,B E A
2,3,B D E
3,4,E C A
4,5,C D B
